# batchnorm-affine-params — worked example 2: Reshape gamma/beta with einops then apply affine

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `batchnorm-affine-params`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The per-channel affine `y = gamma * x_hat + beta` requires the `(C,)` params to broadcast over batch and spatial axes of a `(B, C, H, W)` tensor. `einops.rearrange(gamma, 'c -> 1 c 1 1')` is an explicit, named way to produce the broadcast shape `(1, C, 1, 1)`.

## Worked solution

We want to apply BatchNorm2d's affine step but use **einops** for the reshape instead of `.view`, because the pattern string documents intent.

**Step 1 — write the rearrange pattern.** `rearrange(gamma, 'c -> 1 c 1 1')` reads as: take a 1-axis tensor named `c`, and emit a 4-axis tensor where the singleton axes `1` are inserted before and after. The result has shape `(1, C, 1, 1)`. Same for `beta`.

**Step 2 — confirm the broadcast.** Against `x_hat` of shape `(B, C, H, W)`: `(1, C, 1, 1)` aligns `C` with `C`, and the three size-1 axes stretch over `B`, `H`, `W`. This is exactly the broadcast we need.

**Step 3 — apply the formula.** `g * x_hat + b`. Each spatial location in channel `c` is scaled by `gamma[c]` and shifted by `beta[c]`.

**Why einops here.** The pattern `'c -> 1 c 1 1'` is self-documenting: a reader immediately sees the channel axis is preserved and three singleton axes are added for broadcasting. It is equivalent to `gamma.view(1, -1, 1, 1)` but harder to get silently wrong (e.g. transposing axes).

In [ ]:
def bn_affine_einops(x_hat: Tensor, gamma: Tensor, beta: Tensor) -> Tensor:
    g = rearrange(gamma, 'c -> 1 c 1 1')
    b = rearrange(beta, 'c -> 1 c 1 1')
    return g * x_hat + b

t.manual_seed(0)
B, C, H, W = 2, 4, 3, 3
x_hat = t.randn(B, C, H, W)
gamma = t.tensor([1.5, 1.0, 0.0, 2.0])
beta = t.tensor([0.0, 0.5, 3.0, -1.0])
y = bn_affine_einops(x_hat, gamma, beta)
print(y.shape)
# Channel 2 has gamma=0 -> constant equal to beta[2]=3 everywhere
print(t.allclose(y[:, 2, :, :], t.full((B, H, W), 3.0)))
# Compare against PyTorch's own broadcast reference
ref = gamma.view(1, -1, 1, 1) * x_hat + beta.view(1, -1, 1, 1)
print(t.allclose(y, ref))